### **Installs and Imports**

In [ ]:
!pip install -q transformers datasets peft trl

# Upgrade torchao to a compatible version
!pip install --upgrade torchao

import torch, random
import requests
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


### **Load the base model** and tokenizer, put the model on the GPU.

In [ ]:
model_id = "HuggingFaceTB/SmolLM-135M"     # BASE, not -Instruct

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForCausalLM.from_pretrained(model_id).to(device)

print(f"Loaded {model_id} on {device} — {sum(p.numel() for p in model.parameters()):,} params")

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  538MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loaded HuggingFaceTB/SmolLM-135M on cuda — 134,515,008 params


### **Wrap the Model with LoRA Adapters:**

In [ ]:
lora_config = LoraConfig(
    task_type      = "CAUSAL_LM",                # tells peft this is a next-token LM
    r              = 8,                          # the rank — size of A and B
    lora_alpha     = 16,                         # scaling: effective ΔW = (alpha/r)·B·A
    target_modules = ["q_proj", "v_proj"],       # WHICH layers get adapters
    lora_dropout   = 0.05,                       # dropout on the adapter path
    bias           = "none",                     # don't train bias terms
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


### **CPT-LoRA** using HuggingFace wikitext-dataset:

In [ ]:
ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

# CPT = raw text, every token counts. Join non-empty lines into one corpus.
train_text = "\n".join(t for t in ds["train"]["text"]      if t.strip())
val_text   = "\n".join(t for t in ds["validation"]["text"] if t.strip())

# Pack into one flat stream — exactly your Shakespeare CPT prep, new source
train_ids = torch.tensor(tokenizer(train_text, add_special_tokens=False)["input_ids"])
val_ids   = torch.tensor(tokenizer(val_text,   add_special_tokens=False)["input_ids"])
print(f"train tokens: {len(train_ids):,} | val tokens: {len(val_ids):,}")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

train tokens: 2,543,191 | val tokens: 265,683


### **Hyperparameters + The Batch Loader:**

In [ ]:
block_size, batch_size = 256, 8
lr, weight_decay, grad_clip = 2e-4, 0.01, 1.0
max_steps, eval_every = 500, 50

def get_batch(split):
    d  = train_ids if split == "train" else val_ids
    ix = torch.randint(len(d) - block_size, (batch_size,))
    xb = torch.stack([d[i:i+block_size] for i in ix])
    return xb.to(device)

### **Optimizer** + fixed held-out eval:

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

@torch.no_grad()
def estimate_loss(batches=20):
    model.eval()
    total = 0.0
    for _ in range(batches):
        xb = get_batch("val")
        total += model(input_ids=xb, labels=xb).loss.item()
    model.train()
    return total / batches

### **Training loop:**

In [ ]:
model.train()
for step in range(max_steps):
    xb   = get_batch("train")
    loss = model(input_ids=xb, labels=xb).loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if step % eval_every == 0 or step == max_steps - 1:
        print(f"step {step:4d} | train {loss.item():.4f} | val {estimate_loss():.4f}")

step    0 | train 3.4513 | val 3.4127
step   50 | train 3.1929 | val 3.3479
step  100 | train 3.3043 | val 3.1946
step  150 | train 3.3445 | val 3.1572
step  200 | train 3.1048 | val 3.1717
step  250 | train 2.9079 | val 3.1179
step  300 | train 3.0982 | val 3.0959
step  350 | train 3.0666 | val 3.0314
step  400 | train 2.9533 | val 3.1111
step  450 | train 3.3299 | val 3.0753
step  499 | train 2.8110 | val 3.0682


### **Generate:**

In [ ]:
def sample(prompt, max_new_tokens=120):
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, top_p=0.9, repetition_penalty=1.3,
                         no_repeat_ngram_size=3, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(sample("The history of the Roman Empire"))

The history of the Roman Empire is not well understood. Most scholars believe it was a Greek-speaking city state that existed during the 2nd century BC and continued to exist until AD 508 when Rome became an empire under its emperor Constantine . The first Emperor , before his death in AD 314 , made a pact with the Vandals at the Battle of Adrianople ( Adrianopolis ) on the Danube River where the Romans defeated the invading tribes . The city was then captured by the Vandal Army which forced them out into exile for another decade while the Romans took over as their capital . 

 The city remained


### Before **LoRA-CPT**:

The history of the Roman Empire began in 286 BC, when a large group of people from what is now Turkey were expelled by the Romans. They settled around the city of Rome and became known as the "Romans". The Romans had many different cultures that influenced their way of life - Greek culture was very important to them because it helped shape how they saw themselves today!
One interesting thing about the Romans' relationship with other ancient civilizations like Greece comes up again: there are some similarities between these two groups but also lots more differences too (like language). For example; while Greeks spoke Latin instead of Ancient Greek due its

### After **LoRA-CPT**:

The history of the Roman Empire is a fascinating one. It was not only an empire , it also had its own distinct culture . The Romans were very successful in their conquest and rule over much of Europe until they fell under the power of Germanic tribes at the end of the 5th century AD ; however , as well as some of the smaller empires which arose around this time such as the Byzantine Empire ( which included parts of what are now Turkey, Greece and Bulgaria ) , there has been little historical research on the Romans themselves during this period - despite being thought to have originated from the Roman city of Aquileia in Italy where


### **Save The Model Adapters:**

In [ ]:
model.save_pretrained("smollm-lora-cpt-wikitext")   # saves ONLY the adapters — a few MB, not 500MB



---

## **LoRA-SFT** using Alpaca Datset from HuggingFace:

---



### **Reload and add the adapters:**

In [ ]:
base  = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M").to(device)
model = PeftModel.from_pretrained(base, "smollm-lora-cpt-wikitext", is_trainable=True).to(device)
model.print_trainable_parameters()   # should say ~460,800 trainable — NOT 0

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

trainable params: 460,800 || all params: 134,975,808 || trainable%: 0.3414


### **Load Dataset** (load + format the SFT data (instruction/response pairs)):

In [ ]:
ds = load_dataset("tatsu-lab/alpaca")          # only a 'train' split exists

def format_pair(row):
    instr, inp, out = row["instruction"], row["input"], row["output"]
    if inp.strip():                            # ~40% of rows carry an 'input'
        prompt = f"### Instruction:\n{instr}\n\n### Input:\n{inp}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instr}\n\n### Response:\n"
    return prompt, out

all_pairs = [format_pair(r) for r in ds["train"].select(range(3000))]
random.shuffle(all_pairs)
train_pairs, val_pairs = all_pairs[:2700], all_pairs[2700:]   # our own held-out split
print(f"train pairs: {len(train_pairs)} | val pairs: {len(val_pairs)}")

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…): reconstructing file:   0%|          |  0.00B / 24.2MB            

data/train-00000-of-00001-a09b74b3ef9c3b(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

train pairs: 2700 | val pairs: 300


### **Hyperparameters:**

In [ ]:
EOS = tokenizer.eos_token_id
PAD = tokenizer.pad_token_id
if PAD is None:                              # SmolLM base tokenizer has no pad token
    tokenizer.pad_token = tokenizer.eos_token
    PAD = tokenizer.eos_token_id             # reuse EOS as the pad id
assert EOS is not None and PAD is not None, (EOS, PAD)
print("EOS:", EOS, "| PAD:", PAD)            # confirm both are real ints

MAX_LEN   = 512
batch_sz  = 4
lr, weight_decay, grad_clip = 2e-4, 0.01, 1.0
max_steps, eval_every = 500, 50

EOS: 0 | PAD: 0


### **Masked-example Builder:**

In [ ]:
def build_example(prompt_text, response_text):
    p = tokenizer(prompt_text,   add_special_tokens=False)["input_ids"]
    r = tokenizer(response_text, add_special_tokens=False)["input_ids"] + [EOS]
    input_ids = (p + r)[:MAX_LEN]
    labels    = ([-100]*len(p) + r)[:MAX_LEN]      # mask prompt → loss only on response
    return input_ids, labels

### **Collate SFT Batches (Packing) and Batch-Loader:**

In [ ]:
def collate(batch_pairs):
    ex = [build_example(p, r) for p, r in batch_pairs]
    maxlen = max(len(ids) for ids, _ in ex)
    input_ids, labels, attn = [], [], []
    for ids, lab in ex:
        pad = maxlen - len(ids)
        input_ids.append(ids + [PAD]  * pad)
        labels.append(   lab + [-100] * pad)       # padding never contributes
        attn.append(     [1]*len(ids) + [0]*pad)   # padding mask
    t = lambda z: torch.tensor(z).to(device)
    return t(input_ids), t(labels), t(attn)

def get_sft_batch(pool):
    return collate(random.sample(pool, batch_sz))

### **Optimizer + Fixed held-out Eval:**

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

@torch.no_grad()
def estimate_loss(batches=20):
    model.eval()
    total = 0.0
    for _ in range(batches):
        input_ids, labels, attn = get_sft_batch(val_pairs)
        total += model(input_ids=input_ids, attention_mask=attn, labels=labels).loss.item()
    model.train()
    return total / batches

### **Training Loop** (SFT: masked labels + attention_mask):

In [ ]:
model.train()
for step in range(max_steps):
    input_ids, labels, attn = get_sft_batch(train_pairs)
    loss = model(input_ids=input_ids, attention_mask=attn, labels=labels).loss

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if step % eval_every == 0 or step == max_steps - 1:
        print(f"step {step:4d} | train {loss.item():.4f} | val {estimate_loss():.4f}")

step    0 | train 1.9172 | val 2.1163
step   50 | train 1.9891 | val 1.8349
step  100 | train 1.9418 | val 1.7257
step  150 | train 1.6632 | val 1.7444
step  200 | train 2.0140 | val 1.6770
step  250 | train 1.2146 | val 1.8425
step  300 | train 1.4269 | val 1.7483
step  350 | train 1.7702 | val 1.6493
step  400 | train 1.9155 | val 1.7285
step  450 | train 1.2061 | val 1.7153
step  499 | train 1.6995 | val 1.8017


### **Generate:**

In [ ]:
def sft_generate(instruction, inp=""):
    prompt = (f"### Instruction:\n{instruction}\n\n### Input:\n{inp}\n\n### Response:\n"
              if inp.strip() else
              f"### Instruction:\n{instruction}\n\n### Response:\n")
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    out = model.generate(**enc, max_new_tokens=150, do_sample=True,
                         temperature=0.7, top_p=0.9, repetition_penalty=1.3,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

print(sft_generate("Explain photosynthesis in simple terms."))

Photosynthesis is the process by which plants and some other organisms use sunlight to create food from carbon dioxide, water vapor (in a mixture called "air"), oxygen gas , hydrogen ions . In this way they can survive on their own without needing any external energy source!


### **Base Model:**
Photosynthesis occurs when light energy from the sun is absorbed by chlorophyll molecules within plant cells, resulting in a chemical reaction that turns water and carbon dioxide into glucose (a type of sugar) through an electron transfer process called photophosphorylation or chemiosmosis. This unique method allows plants to convert sunlight directly using their own internal capacity for producing ATP—the powerhouse responsible for generating electricity! By doing so efficiently while maintaining stability inside cell walls like those found on leaves; chloroplasts play essential roles during growth cycles involving multiple stages such as germination/ovulation phases followed closely after fertilization until reaching maturity level where reproduction takes place via sexual means including pollination between male and female reproductive organs involved here before releasing fertilized eggs carrying genetic information necessary later forming new individuals


### **After CPT:**
1 . The answer is a true statement

2  In the process of converting sunlight into food through light-dependent reactions , plants use energy from molecules and chemical bonds to convert carbon dioxide ( CO₂ ) present within air or water vapor onto glucose molecule which then can be used as fuel for cellular respiration by living cells

3 . In order to get more oxygen required during combustion, organisms have evolved specialised structures such as leaves that capture solar radiation at night whilst trapping it inside their bodies while allowing animals consuming them access daily without any risk if they were not able to obtain sufficient amounts needed throughout day time when less efficient mechanisms are available like photolysis [ 9 ] / photosynthesis ; these organs enable greater efficiency towards obtaining O₃ gas necessary via chem



### **After SFT:**
Photosynthesis is the process by which plants and other photosynthetic organisms convert light energy into chemical potential from water to produce glucose (sugar) as a source of fuel for growth, development ,and maintenance .

### **Save The Adapters:**

In [ ]:
model.save_pretrained("smollm-lora-sft-alpaca")



---

## **RLHF + DPO (Direct Preference Optimization)**

---



### **Rebuild the SFT'd model and MERGE the SFT adapters into the base:**

`merge_and_unload` computes `W' = W + (alpha/r)·B·A` for every adapted layer and returns a plain model with no LoRA machinery left.

In [ ]:
base  = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M").to(device)
sft   = PeftModel.from_pretrained(base, "smollm-lora-sft-alpaca")   # reattach SFT adapters
model = sft.merge_and_unload()                                      # fold W + (α/r)·B·A → W
# `model` is now an ORDINARY CausalLM whose weights ARE the SFT'd SmolLM

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

### **Load + Format the preference dataset** (standard prompt/chosen/rejected):

In [ ]:
TEMPLATE = "### Instruction:\n{instruction}\n\n### Response:\n"   # SAME template as SFT

raw = load_dataset("Intel/orca_dpo_pairs", split="train")
print(raw[0].keys())   # dict_keys(['system', 'question', 'chosen', 'rejected'])

def to_pref(row):
    return {
        "prompt":   TEMPLATE.format(instruction=row["question"]),
        "chosen":   row["chosen"],
        "rejected": row["rejected"],
    }

pref = raw.map(to_pref, remove_columns=raw.column_names).select(range(2000))
print(pref[0]["prompt"][:120])
print("CHOSEN  :", pref[0]["chosen"][:80])
print("REJECTED:", pref[0]["rejected"][:80])

README.md:   0%|          | 0.00/196 [00:00<?, ?B/s]

orca_rlhf.jsonl: reconstructing file:   0%|          |  0.00B / 36.3MB            

orca_rlhf.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/12859 [00:00<?, ? examples/s]

dict_keys(['system', 'question', 'chosen', 'rejected'])


Map:   0%|          | 0/12859 [00:00<?, ? examples/s]

### Instruction:
You will be given a definition of a task first, then some input of the task.
This task is about using t
CHOSEN  : [
  ["AFC Ajax (amateurs)", "has ground", "Sportpark De Toekomst"],
  ["Ajax You
REJECTED:  Sure, I'd be happy to help! Here are the RDF triplets for the input sentence:




### **LoRA config for the DPO stage + The DPO Hyperparameters:**

In [ ]:
peft_config = LoraConfig(
    task_type="CAUSAL_LM", r=8, lora_alpha=16,
    target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none",
)

dpo_config = DPOConfig(
    output_dir                  = "smollm-dpo-orca",
    beta                        = 0.1,        # β — the KL-leash strength from the DPO loss
    learning_rate               = 5e-6,       # tiny, for the reasons from last session
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,          # effective batch = 2 × 4 = 8
    num_train_epochs            = 1,
    max_steps                   = 300,        # cap for a fast Colab run
    max_length                  = 1024,
    warmup_steps                = 0.1,
    lr_scheduler_type           = "cosine",
    logging_steps               = 20,
    bf16                        = torch.cuda.is_available(),
    report_to                   = "none",
)

### **Build the DPOTrainer and Train:**

In [ ]:
trainer = DPOTrainer(
    model           = model,          # the merged SFT model (plain CausalLM)
    ref_model       = None,           # None → reference = this model with the DPO adapter DISABLED
    args            = dpo_config,
    train_dataset   = pref,
    processing_class= tokenizer,      # TRL's current name for the tokenizer argument
    peft_config     = peft_config,    # fresh adapters, added on top of the merged SFT base
)
trainer.train()

Adding EOS to train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
20,0.700001
40,0.689782
60,0.691015
80,0.684362
100,0.688669
120,0.676060
140,0.685287
160,0.688713
180,0.675625
200,0.673475


TrainOutput(global_step=300, training_loss=0.6791325791676839, metrics={'train_runtime': 1948.704, 'train_samples_per_second': 1.232, 'train_steps_per_second': 0.154, 'total_flos': 1785088616085504.0, 'train_loss': 0.6791325791676839, 'epoch': 1.2152284263959392})

### **Show Log History:**

In [ ]:
import pandas as pd
logs = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ["step","loss","rewards/accuracies","rewards/margins",
                    "rewards/chosen","rewards/rejected"] if c in logs]
print(logs[cols].dropna().to_string(index=False))

 step     loss  rewards/accuracies  rewards/margins  rewards/chosen  rewards/rejected
   20 0.700001            0.437500        -0.010935       -0.000214          0.010721
   40 0.689782            0.550000         0.008992       -0.000208         -0.009200
   60 0.691015            0.500000         0.006294       -0.000015         -0.006309
   80 0.684362            0.543750         0.020193        0.004586         -0.015607
  100 0.688669            0.562500         0.011755       -0.013304         -0.025059
  120 0.676060            0.643750         0.037475        0.007229         -0.030247
  140 0.685287            0.581250         0.018409       -0.017190         -0.035598
  160 0.688713            0.556250         0.012185       -0.024919         -0.037104
  180 0.675625            0.656250         0.037959       -0.020069         -0.058028
  200 0.673475            0.637500         0.044354       -0.025054         -0.069407
  220 0.664831            0.693750         0.060667   

### **Save the DPO adapter:**

In [ ]:
trainer.save_model("smollm-dpo-orca")

### **Generate & Compare:**

In [ ]:
def compare(instruction, max_new_tokens=150, seed=0):
    prompt = TEMPLATE.format(instruction=instruction)
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    m = trainer.model; m.eval()
    kw = dict(max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7,
              top_p=0.9, repetition_penalty=1.3, pad_token_id=tokenizer.eos_token_id)
    strip = lambda o: tokenizer.decode(o[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)

    torch.manual_seed(seed)                          # same random draw for a fair A/B
    with torch.no_grad(), m.disable_adapter():       # DPO adapter OFF → SFT model
        before = strip(m.generate(**enc, **kw))

    torch.manual_seed(seed)                          # reseed → identical sampling stream
    with torch.no_grad():                            # DPO adapter ON → aligned model
        after = strip(m.generate(**enc, **kw))

    print("PROMPT:", instruction)
    print("\n── SFT (adapter OFF, before DPO) ──\n", before)
    print("\n── DPO (adapter ON, after) ──\n", after)

compare("Explain why the sky is blue.")

PROMPT: Explain why the sky is blue.

── SFT (adapter OFF, before DPO) ──
 The reason for the color of skies and sunsets being a beautiful shade that we see during sunset or sunrise, as opposed to other colors such as red because it takes longer time than in winter due to its higher temperature level which makes us perceive more blues from this perspective

── DPO (adapter ON, after) ──
 The reason for the color of skies being seen as they appear to be clear and bluish due to low atmospheric concentrations, which results in a decrease in red light intensity over time because it takes longer at higher altitudes than lower latitudes such as Antarctica or high mountains where sunlight can reach more directly through clouds that are visible from above them (a phenomenon known as the albedo effect).


### **PROMPT**: Explain why the sky is blue.

### ── **SFT (adapter OFF, before DPO)** ──
 The reason for the color of skies and sunsets being a beautiful shade that we see during sunset or sunrise, as opposed to other colors such as red because it takes longer time than in winter due to its higher temperature level which makes us perceive more blues from this perspective

### ── **DPO (adapter ON, after)** ──
 The reason for the color of skies being seen as they appear to be clear and bluish due to low atmospheric concentrations, which results in a decrease in red light intensity over time because it takes longer at higher altitudes than lower latitudes such as Antarctica or high mountains where sunlight can reach more directly through clouds that are visible from above them (a phenomenon known as the albedo effect).
